In [0]:
import pytest
from pyspark.testing.utils import assertSchemaEqual, assertDataFrameEqual
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [0]:
SCHEMA_DB = "training_bs.user_uenishi"
SCHEMA_SAS = "training_bs.user_uenishi_sas"

tables = [row.name for row in spark.catalog.listTables(SCHEMA_DB)]

In [0]:
class TestImportSourceFiles:
    """import-source-files の取り込み結果を検証するテスト"""

    @pytest.mark.parametrize("table", tables)
    def test_schema_equal(self, table):
        """スキーマのカラム名・データ型が一致すること"""
        sdf_db = spark.read.table(f"{SCHEMA_DB}.{table}")
        sdf_sas = spark.read.table(f"{SCHEMA_SAS}.{table}")
        assertSchemaEqual(sdf_db.schema, sdf_sas.schema)

    @pytest.mark.parametrize("table", tables)
    def test_count_equal(self, table):
        """レコード件数が一致すること"""
        sdf_db = spark.read.table(f"{SCHEMA_DB}.{table}")
        sdf_sas = spark.read.table(f"{SCHEMA_SAS}.{table}")
        assert sdf_db.count() == sdf_sas.count(), \
            f"テーブル {table}: DB={sdf_db.count()}, SAS={sdf_sas.count()}"

    @pytest.mark.parametrize("table", tables)
    def test_data_equal(self, table):
        """全レコードの値が完全一致すること"""
        sdf_db = spark.read.table(f"{SCHEMA_DB}.{table}")
        sdf_sas = spark.read.table(f"{SCHEMA_SAS}.{table}")
        assertDataFrameEqual(sdf_db, sdf_sas)